In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# トランスパイラーのための量子コンピューターの表現


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

抽象的な回路を特定の QPU（量子処理ユニット）で実行できる ISA 回路に変換するには、トランスパイラーが QPU に関する特定の情報を必要とします。この情報は 2 つの場所にあります：ジョブを送信する予定の `BackendV2`（またはレガシーの `BackendV1`）オブジェクトと、バックエンドの `Target` 属性です。

- [`Target`](../api/qiskit/qiskit.transpiler.Target) には、ネイティブ基底ゲート、量子ビット接続性、パルスやタイミング情報など、デバイスに関するすべての関連制約が含まれています。
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) はデフォルトで `Target` を持ち、[`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap) などの追加情報を含み、量子回路ジョブを送信するためのインターフェースを提供します。

また、特定のユースケースがある場合や、この情報がトランスパイラーによるより最適化された回路の生成に役立つと考える場合は、トランスパイラーが使用する情報を明示的に提供することもできます。

特定のハードウェアに対してトランスパイラーが最適な回路を生成する精度は、`Target` または `Backend` が保持する制約情報の量に依存します。

> **Note:** 基礎となるトランスパイルアルゴリズムの多くは確率的であるため、より良い回路が必ずしも見つかるとは限りません。

このページでは、QPU 情報をトランスパイラーに渡す例をいくつか示します。これらの例では、[`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) モックバックエンドのターゲットを使用します。

<span id="default-config"></span>
## デフォルト設定
トランスパイラーの最も簡単な使い方は、`Backend` または `Target` を提供することで、すべての QPU 情報を提供することです。トランスパイラーの動作をより深く理解するために、回路を構築し、以下のように異なる情報でトランスパイルします。

必要なライブラリをインポートし、QPU をインスタンス化します：
特定のプロセッサーで実行できる ISA 回路に抽象的な回路を変換するには、トランスパイラーがプロセッサーに関する特定の情報を必要とします。通常、この情報はトランスパイラーに提供される [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) または [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) に格納されており、追加情報は必要ありません。ただし、特定のユースケースがある場合や、この情報がトランスパイラーによるより最適化された回路の生成に役立つと考える場合は、トランスパイラーが使用する情報を明示的に提供することもできます。

このトピックでは、トランスパイラーに情報を渡す例をいくつか示します。これらの例では、[`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) モックバックエンドのターゲットを使用します。

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

サンプル回路には、Qiskit の回路ライブラリから [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) のインスタンスを使用します。

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

この例では、デフォルト設定を使用して `backend` の `target` にトランスパイルします。これにより、回路をバックエンドで実行できる回路に変換するために必要なすべての情報が提供されます。

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

この例は、このトピックの後続のセクションで、カップリングマップと基底ゲートが最適な回路構築のためにトランスパイラーに渡すべき重要な情報であることを示すために使用されます。QPU は通常、タイミングやスケジューリングなど、渡されなかった他の情報についてはデフォルト設定を選択できます。

## カップリングマップ

カップリングマップは、どの量子ビットが接続されており、したがってそれらの間に 2 量子ビットゲートが存在するかを示すグラフです。このグラフが方向性を持つ場合があり、その場合 2 量子ビットゲートは一方向にしか作用できません。ただし、トランスパイラーは追加の 1 量子ビットゲートを加えることでゲートの方向を常に反転できます。抽象的な量子回路は、接続性が限られていても、量子情報を移動させるための SWAP ゲートを導入することで、常にこのグラフ上に表現できます。

抽象回路の量子ビットを _仮想量子ビット_、カップリングマップ上の量子ビットを _物理量子ビット_ と呼びます。トランスパイラーは仮想量子ビットと物理量子ビットの間のマッピングを提供します。トランスパイルの最初のステップの一つである _レイアウト_ ステージがこのマッピングを実行します。

> **Note:** ルーティングステージは、実際の量子ビットを選択する _レイアウト_ ステージと密接に関連していますが、デフォルトでは、このトピックでは簡略化のためにそれらを別々のステージとして扱います。ルーティングとレイアウトの組み合わせは _量子ビットマッピング_ と呼ばれます。これらのステージの詳細については、[トランスパイラーステージ](transpiler-stages)のトピックを参照してください。

`coupling_map` キーワード引数を渡して、トランスパイラーへの影響を確認します：

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

上記に示すように、いくつかの SWAP ゲートが挿入されています（それぞれ 3 つの CX ゲートで構成されています）。これは現在のデバイスでは多くのエラーを引き起こします。実際の量子ビットトポロジー上でどの量子ビットが選択されているかを確認するには、Qiskit Visualizations の `plot_circuit_layout` を使用します：

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

これにより、仮想量子ビット 0〜11 が物理量子ビット 0〜11 のラインに自明的にマッピングされたことがわかります。ルーティングが必要な場合に `VF2Layout` を使用するデフォルト（`optimization_level=1`）に戻りましょう。

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

SWAP ゲートが挿入されなくなり、選択された物理量子ビットは `target` クラスを使用した場合と同じになりました。

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

レイアウトがリング状になりました。このレイアウトは回路の接続性を尊重しているため、SWAP ゲートが存在せず、実行に対してはるかに優れた回路が得られます。

## 基底ゲート

すべての量子コンピューターは _基底ゲート_ と呼ばれる限られた命令セットをサポートしています。回路内のすべてのゲートは、このセットの要素に変換される必要があります。このセットは、任意の量子演算をそれらのゲートに分解できることを意味するユニバーサルゲートセットを提供する、1 量子ビットおよび 2 量子ビットゲートで構成される必要があります。これは [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator) によって行われ、基底ゲートはこの情報を提供するためにトランスパイラーへのキーワード引数として指定できます。

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

_ibm_sherbrooke_ のデフォルトの 1 量子ビットゲートは `rz`、`x`、`sx` で、デフォルトの 2 量子ビットゲートは `ecr`（エコーされたクロスレゾナンス）です。CX ゲートは `ecr` ゲートから構成されるため、一部の QPU では `ecr` が 2 量子ビット基底ゲートとして指定され、他の QPU では `cx` がデフォルトです。`ecr` ゲートは `cx` ゲートの _エンタングリング_ 部分です。制御ゲートに加えて、`delay` および `measurement` 命令もあります。

> **Note:** QPU にはデフォルトの基底ゲートがありますが、命令を提供するかパルスゲートを追加する限り（[カスタムトランスパイラーパスの作成](custom-transpiler-pass)を参照）、任意のゲートを選択できます。デフォルトの基底ゲートは QPU でキャリブレーションが行われたゲートであるため、追加の命令/パルスゲートを提供する必要はありません。たとえば、一部の QPU では `cx` がデフォルトの 2 量子ビットゲートであり、他の QPU では `ecr` です。詳細については、[ネイティブゲートと演算](/guides/qpu-information#native-gates)の一覧を参照してください。

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

`CXGate` オブジェクトが `ecr` ゲートと 1 量子ビット基底ゲートに分解されたことに注目してください。

## デバイスエラーレート
`Target` クラスには、デバイス上の演算のエラーレートに関する情報を含めることができます。
例えば、以下のコードは量子ビット 1 と 0 の間のエコーされたクロスレゾナンス（ECR）ゲートのプロパティを取得します（ECR ゲートは方向性があることに注意してください）：

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

出力にはゲートの持続時間（秒単位）とエラーレートが表示されます。エラー情報をトランスパイラーに開示するには、上記の `basis_gates` と `coupling_map` を使ってターゲットモデルを構築し、バックエンド `FakeSherbrooke` からのエラー値を設定します。